In [1]:
import numpy as np
import faiss


d = 256                          # размерность векторов
nb = 100000                      # число векторов (мы их создадим случайно в таком количестве)
nq = 10000                       # число векторов для поиска

np.random.seed(1234)            
xb = np.random.random((nb, d)).astype('float32')
xb[:, 0] += np.arange(nb) / 1000. # добавляем информацию об очерёдности векторов

index = faiss.IndexFlatL2(d)   # инициализируем индекс
print(index.is_trained)
index.add(xb)                  # добавляем векторы
print(index.ntotal)

True
100000


In [5]:
def get_index_size(index):
    import os
    
    # запишем индекс на диск
    faiss.write_index(index, 'data/temp.index')
    # получаем размер файла
    index_size = os.path.getsize('data/temp.index')
    # удаляем сохранённый индекс
    os.remove('data/temp.index')
    return index_size


# инициализируем и вычисляем размер l2 индекса
index_l2 = faiss.IndexFlatL2(d)
index_l2.add(xb)    
index_l2_size = get_index_size(index_l2)

# инициализируем и вычисляем размер PQ индекса
M = 16
assert d % M == 0 # из исходного вектора должно получаться целое число векторов
nbits = 8

index_pq = faiss.IndexPQ(d, M, nbits)
index_pq.train(xb) # обучаем индекс PQ
index_pq_size = get_index_size(index_pq)

print(f"Отношение индексов PQ/L2: {index_pq_size/index_l2_size:.4f}")

Отношение индексов PQ/L2: 0.0026
